## df = spark.read.format("delta").table("schema.tabela")

Gerando um dataframe dos delta lake no container bronze do Azure Data Lake Storage

In [0]:
df_apolice   = spark.read.format("delta").table("silver.apolice")
df_carro     = spark.read.format("delta").table("silver.carro")
df_cliente   = spark.read.format("delta").table("silver.cliente")
df_endereco  = spark.read.format("delta").table("silver.endereco")
df_estado    = spark.read.format("delta").table("silver.estado")
df_marca     = spark.read.format("delta").table("silver.marca")
df_modelo    = spark.read.format("delta").table("silver.modelo")
df_municipio = spark.read.format("delta").table("silver.municipio")
df_regiao    = spark.read.format("delta").table("silver.regiao")
df_sinistro  = spark.read.format("delta").table("silver.sinistro")
df_telefone  = spark.read.format("delta").table("silver.telefone")


### Adicionando metadados de data e hora de processamento e nome do arquivo de origem

In [0]:
%sql
drop table if exists gold.dim_carro

In [0]:
%sql
create table gold.dim_carro (
  SK_CARRO             bigint generated by default as identity,
  PLACA                varchar(10),
  MARCA                varchar(100),
  MODELO               varchar(100),
  COR                  varchar(10),
  ANO                  int,
  CHASSI               varchar(20)
)
USING DELTA;

In [0]:
%sql
DESCRIBE TABLE EXTENDED gold.dim_carro

In [0]:
df_modelo.createOrReplaceTempView("modelo")
df_marca.createOrReplaceTempView("marca")
df_carro.createOrReplaceTempView("carro")

In [0]:
%sql

WITH carros_relacional AS (
    SELECT c.placa,        
           ma.nome_marca,   
           mo.nome_modelo,  
           c.cor,           
           c.ano,           
           c.chassi
      FROM carro c
           INNER JOIN modelo mo
             ON c.codigo_modelo = mo.codigo_modelo
           INNER JOIN marca ma
             ON mo.codigo_marca = ma.codigo_marca
)
MERGE INTO
	gold.dim_carro AS c
USING
	carros_relacional AS cc
ON c.placa = cc.placa   

WHEN MATCHED AND (c.marca <> cc.nome_marca OR c.modelo <> cc.nome_modelo OR c.cor <> cc.cor OR c.ano <> cc.ano OR c.chassi <> cc.chassi) THEN

	UPDATE SET placa  = cc.placa,
	           marca  = cc.nome_marca,
			   modelo = cc.nome_modelo,
			   cor    = cc.cor,
			   ano    = cc.ano,
			   chassi = cc.chassi

WHEN NOT MATCHED THEN
	INSERT (placa, marca, modelo, cor, ano, chassi)
	VALUES (cc.placa, cc.nome_marca, cc.nome_modelo, cc.cor, cc.ano, cc.chassi)


In [0]:
%sql
select * from gold.dim_carro

In [0]:
%sql
drop table if exists gold.dim_tempo

In [0]:
from pyspark.sql.functions import expr, date_format

# Define o intervalo de datas desejado
data_inicial = "2023-01-01"
data_final = "2026-12-31"

# Calcula o número de dias no intervalo
num_dias = spark.sql(f"SELECT datediff('{data_final}', '{data_inicial}')").collect()[0][0]

# Cria um DataFrame com uma coluna contendo uma sequência de datas
df_calendario = spark.range(0, num_dias + 1) \
    .selectExpr(f"date_add(to_date('{data_inicial}'), CAST(id AS INT)) AS Data")

# Extrai os componentes de data
df_tempo = df_calendario.selectExpr(
    "Data",
    "year(Data) AS Ano",
    "month(Data) AS Mes",
       "(CASE month(Data) \
        WHEN 1 THEN 'JANEIRO' \
        WHEN 2 THEN 'FEVEREIRO' \
        WHEN 3 THEN 'MARCO' \
        WHEN 4 THEN 'ABRIL' \
        WHEN 5 THEN 'MAIO' \
        WHEN 6 THEN 'JUNHO' \
        WHEN 7 THEN 'JULHO' \
        WHEN 8 THEN 'AGOSTO' \
        WHEN 9 THEN 'SETEMBRO' \
        WHEN 10 THEN 'OUTUBRO' \
        WHEN 11 THEN 'NOVEMBRO' \
        WHEN 12 THEN 'DEZEMBRO' \
    END) AS NomeMes",
    "day(Data) AS Dia",
    "(CASE dayofweek(Data) \
        WHEN 1 THEN 'DOMINGO' \
        WHEN 2 THEN 'SEGUNDA-FEIRA' \
        WHEN 3 THEN 'TERCA-FEIRA' \
        WHEN 4 THEN 'QUARTA-FEIRA' \
        WHEN 5 THEN 'QUINTA-FEIRA' \
        WHEN 6 THEN 'SEXTA-FEIRA' \
        WHEN 7 THEN 'SABADO' \
    END) AS NomeDiaSemana",
    "dayofweek(Data) AS NumeroDiaSemana"
)

# Exibe o DataFrame resultante
df_tempo.display()

df_tempo.write.mode('overwrite').saveAsTable("gold.dim_tempo", format="delta")


In [0]:
%sql
drop table if exists gold.dim_cliente

In [0]:
%sql
create table gold.dim_cliente (
   SK_CLIENTE           bigint generated by default as identity,
   CODIGO_CLIENTE       int,
   NOME                 varchar(50),
   CPF                  varchar(11),
   SEXO                 char(1),
   DATA_NASCIMENTO      date
)
USING DELTA;


In [0]:
df_cliente.createOrReplaceTempView("cliente")

In [0]:
%sql
--v3a (COM CTE) prevendo atualizacao SCD1
WITH cliente_relacional AS (
	SELECT codigo_cliente,
		   nome,
		   cpf,
		   sexo,
		   data_nascimento
	  FROM cliente
)
MERGE INTO
	gold.dim_cliente AS d
USING
	cliente_relacional AS r
ON r.codigo_cliente = d.codigo_cliente   

WHEN MATCHED AND (r.codigo_cliente <> d.codigo_cliente OR r.nome <> d.nome OR r.cpf <> d.cpf OR r.sexo <> d.sexo OR r.data_nascimento <> d.data_nascimento) THEN

	UPDATE SET codigo_cliente    = r.codigo_cliente,
	           nome          = r.nome,
			   cpf           = r.cpf,
			   sexo          = r.sexo,
			   data_nascimento = r.data_nascimento

WHEN NOT MATCHED THEN

	INSERT (codigo_cliente, nome, cpf, sexo, data_nascimento)
	VALUES (r.codigo_cliente, r.nome, r.cpf, r.sexo, r.data_nascimento)



In [0]:
%sql
select * from gold.dim_cliente

In [0]:
%sql
drop table if exists gold.dim_localidade

In [0]:
%sql

create table gold.dim_localidade (
   SK_LOCALIDADE        bigint generated by default as identity,
   CODIGO_MUNICIPIO     int,
   NOME_MUNICIPIO       varchar(100),
   NOME_ESTADO          varchar(100),
   NOME_REGIAO          varchar(100)
)
USING DELTA

In [0]:
df_municipio.createOrReplaceTempView("municipio")
df_estado.createOrReplaceTempView("estado")
df_regiao.createOrReplaceTempView("regiao")

In [0]:
%sql
--v3a (COM CTE) prevendo atualizacao SCD1
WITH localidade_relacional AS (
	SELECT codigo_municipio,
		   nome_municipio,
		   nome_estado,
		   nome_regiao
	  FROM municipio m
		   INNER JOIN estado e
		     ON m.codigo_estado = e.codigo_estado
		   INNER JOIN regiao r
		     ON e.codigo_regiao = r.codigo_regiao
)
MERGE INTO
	gold.dim_localidade AS d
USING
	localidade_relacional AS r

ON r.codigo_municipio = d.codigo_municipio

WHEN MATCHED AND (r.codigo_municipio <> d.codigo_municipio OR r.nome_municipio <> d.nome_municipio OR r.nome_estado <> d.nome_estado OR r.nome_regiao <> d.nome_regiao) THEN
	UPDATE SET codigo_municipio = r.codigo_municipio,
		       nome_municipio = r.nome_municipio,
			   nome_estado    = r.nome_estado,
		       nome_regiao    = r.nome_regiao

WHEN NOT MATCHED THEN
	INSERT (codigo_municipio, nome_municipio, nome_estado, nome_regiao)
	VALUES (r.codigo_municipio, r.nome_municipio, r.nome_estado, r.nome_regiao)


In [0]:
%sql
select * from gold.dim_localidade

In [0]:
%sql
drop table if exists gold.fato_sinistro

In [0]:
%sql
create table gold.fato_sinistro (
   FK_TEMPO             date,
   FK_LOCALIDADE        int,
   FK_CARRO             int,
   FK_CLIENTE           int,
   QTDE_SINISTRO        int
)
USING DELTA;

In [0]:
df_apolice.createOrReplaceTempView("apolice")
df_sinistro.createOrReplaceTempView("sinistro")

In [0]:
%sql
with apolice_cliente as (
	select a.codigo_cliente, placa from apolice a inner join cliente c on a.codigo_cliente = c.codigo_cliente
)
insert into gold.fato_sinistro
select data,
       sk_localidade,
	   sk_carro,
	   sk_cliente,
	   count(1) as qtde_sinistro
from sinistro r
       inner join gold.dim_carro dcar
	     on r.placa = dcar.placa
	   inner join apolice_cliente ac
	     on ac.placa = r.placa
	   inner join gold.dim_cliente dcli
	     on ac.codigo_cliente = dcli.codigo_cliente
	   inner join gold.dim_localidade dloc
	     on r.local_sinistro = dloc.codigo_municipio
	   inner join gold.dim_tempo dtem
	     on r.data_sinistro = dtem.data
group by data,
       sk_localidade,
	   sk_carro,
	   sk_cliente

In [0]:
%sql
select * from gold.fato_sinistro